# Edge Associations With Covariates

## Notebook Purpose

Use this notebook to run the edge-association pipeline:
1. Configure paths and options
2. Load and prepare cohorts
3. Build edge feature matrices for depression/control groups
4. Run per-edge associations by covariate
5. Save and inspect summary outputs

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

# Ensure source_code/connectivity_matrices is importable
repo_root = Path.cwd()
module_dir = repo_root / 'source_code' / 'connectivity_matrices'
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from edge_associations_utils import (
    get_motion_columns,
    build_edge_feature_matrix_from_connectivity,
    run_per_edge_associations,
    describe_significant_edges,
    load_and_prepare_cohort_data,
)

## Configuration

Configure following parameters.

In [ ]:
# Configuration
CONNECTIVITY_TYPES = ['functional', 'structural']
EDGE_COVARIATES = ['age', 'motion', 'sex']

GENERAL_DIR = '.../data/UKB/cohorts'
DEPRESSION_DIR = '.../data/UKB/F32_notask_STRCO_RSSCHA_RSTIA'
CONTROL_DIR = '.../data/UKB/control_notask_STRCO_RSSCHA_RSTIA'
COMBINED_COHORT_PATH = '.../data/UKB/cohorts/combined_cohort_F32.csv'
DEPRESSION_COHORT_PATH = '.../data/UKB/cohorts/depression_cohort_F32.csv'
HEAD_MOTION_PATH = '.../data/UKB/head_motion.csv.gz'
VALIDATION_PLOTS_BASE_DIR = '.../reports/plots/schaefer1000+tian54'

fMRI_MOTION_METRIC = 'p24441_i2'
dMRI_MOTION_METRIC = 'p24453_i2'
BATCH_SIZE = int(os.getenv('GC_BATCH_SIZE', '25'))

os.makedirs(GENERAL_DIR, exist_ok=True)
print(f'Using batch size: {BATCH_SIZE}')
print('Configured modalities:', CONNECTIVITY_TYPES)

## Step 1 — Load and Prepare Cohort Data

In [ ]:
cohort_data = load_and_prepare_cohort_data(
    COMBINED_COHORT_PATH,
    DEPRESSION_COHORT_PATH,
    HEAD_MOTION_PATH,
    save_if_modified=True,
)

print('Depression subjects:', len(cohort_data['depression_subject_ids']))
print('Control subjects   :', len(cohort_data['control_subject_ids']))

## Step 2 — Define Per-Modality Execution Function

This function encapsulates the core logic for running the edge-association pipeline for a single connectivity type, including:
- Building edge feature matrices
- Running per-edge associations
- Saving outputs

In [ ]:
def execute_edge_associations_for_conn_type(conn_type, cohort_data, edge_covariates):
    combined_data = cohort_data['combined_data']
    depression_subject_ids = list(cohort_data['depression_subject_ids'])
    control_subject_ids = list(cohort_data['control_subject_ids'])

    validation_plots_dir = os.path.join(VALIDATION_PLOTS_BASE_DIR, f'{conn_type}_con')
    os.makedirs(validation_plots_dir, exist_ok=True)

    motion_columns = get_motion_columns(conn_type, fMRI_MOTION_METRIC, dMRI_MOTION_METRIC)
    primary_motion_metric = next(iter(motion_columns.values()))

    print(f'\n[STEP 2] Loading {conn_type} connectivity data...')
    X_dep, n_nodes, dep_ids = build_edge_feature_matrix_from_connectivity(
        depression_subject_ids,
        DEPRESSION_DIR,
        conn_type,
        batch_size=BATCH_SIZE,
        prefix=f'{conn_type}_dep_edges',
        dtype=np.float32,
    )
    X_ctrl, n_nodes_ctrl, ctrl_ids = build_edge_feature_matrix_from_connectivity(
        control_subject_ids,
        CONTROL_DIR,
        conn_type,
        batch_size=BATCH_SIZE,
        prefix=f'{conn_type}_ctrl_edges',
        dtype=np.float32,
    )

    if X_dep is None or X_ctrl is None:
        print('  No connectivity matrices found for this modality; skipping.')
        return

    if n_nodes != n_nodes_ctrl:
        raise ValueError(f'Node count mismatch dep={n_nodes} ctrl={n_nodes_ctrl}')

    depression_subject_ids = dep_ids
    control_subject_ids = ctrl_ids

    print(f'  Loaded edge vectors for {X_dep.shape[0]} depressed subjects')
    print(f'  Loaded edge vectors for {X_ctrl.shape[0]} control subjects')

    print('\n[STEP 3] Edge association analysis...')
    results = run_per_edge_associations(
        X_dep, X_ctrl, combined_data, depression_subject_ids, control_subject_ids,
        conn_type, validation_plots_dir, covariates=edge_covariates,
        motion_metric=primary_motion_metric, motion_metrics=motion_columns,
        analysis_level='matrix_edges', connectivity_metric='edge weight',
    )

    print('Number of edges FDR significantly associated with covariates:')
    os.makedirs(GENERAL_DIR, exist_ok=True)

    for cov in edge_covariates:
        print(f'  Covariate: {cov}')

        if cov == 'motion':
            cov = motion_columns.keys() # because only one motion metric stored per modality
            print(f"    Using motion metric: {cov}")

        control_corr_map = results[f'{cov}_ctrl_map']
        depression_corr_map = results[f'{cov}_dep_map']
        count_control, percentage_control, rmin_control, rmax_control, median_r_control = describe_significant_edges(control_corr_map)
        count_depressed, percentage_depressed, rmin_depressed, rmax_depressed, median_r_depressed = describe_significant_edges(depression_corr_map)

        fname = f'{conn_type}_{cov}_edge_association.txt'
        out_path = os.path.join(GENERAL_DIR, fname)
        with open(out_path, 'w') as txt:
            txt.write('conn_type;covariate;cohort;count;percentage;rmin;rmax;median\n')
            txt.write(
                f'{conn_type};{cov};Control;'
                f'{int(count_control)};{percentage_control:.2f};'
                f'{rmin_control:.3f};{rmax_control:.3f};{median_r_control:.3f}\n'
            )
            txt.write(
                f'{conn_type};{cov};Depression;'
                f'{int(count_depressed)};{percentage_depressed:.2f};'
                f'{rmin_depressed:.3f};{rmax_depressed:.3f};{median_r_depressed:.3f}\n'
            )

        print(f'    [{cov}] Control   : {count_control} edges ({percentage_control:.2f}%), range [{rmin_control:.3f}, {rmax_control:.3f}], median {median_r_control:.3f}')
        print(f'    [{cov}] Depression: {count_depressed} edges ({percentage_depressed:.2f}%), range [{rmin_depressed:.3f}, {rmax_depressed:.3f}], median {median_r_depressed:.3f}')
       

## Step 3 — Execute the Pipeline Across Modalities

This runs the full edge-association pipeline for each connectivity type, saving outputs to the specified directory.

In [ ]:
print('=' * 80)
print('EDGE ASSOCIATION PIPELINE')
print(f"Modalities queued: {', '.join(CONNECTIVITY_TYPES)}")
print('=' * 80)

for conn_type in CONNECTIVITY_TYPES:
    execute_edge_associations_for_conn_type(conn_type, cohort_data, EDGE_COVARIATES)

print('\n' + '=' * 80)
print('EDGE ASSOCIATION PIPELINE COMPLETE')
print('=' * 80)

## Step 4 — Quick Output Inspection

This optional cell lists generated summary files so you can quickly confirm outputs.

In [ ]:
from pathlib import Path

summary_files = sorted(Path(GENERAL_DIR).glob('*_edge_association.txt'))
plot_dirs = [Path(VALIDATION_PLOTS_BASE_DIR) / f'{ct}_con' for ct in CONNECTIVITY_TYPES]

print('Summary files:')
for path in summary_files:
    print(' -', path)

print('\nValidation plot directories:')
for d in plot_dirs:
    print(' -', d, '(exists=' + str(d.exists()) + ')')